# Calculate and visualize reference activation levels for WM, Emotion, and Language tasks

For information on how to use this notebook along with the original `.ipynb` file, please check the corresponding [Python notebook in the preprocessing reports repository from Dr. Raúl Rodriguez Cruces](https://github.com/rcruces/preproc_reports/blob/main/notebooks/2024_neurosynth-ICBM152-fsLR.ipynb).

Please note that a significant part of this notebook is different from the original version by Dr. Raúl Rodriguez Cruces. Most of the code is set in a way that eliminates using containers and heavy software.

## Setup
The MNI atlases are stored in `./ref_activation_ns/data/` and Neurosynth `association-test_z_FDR_0.01.nii.gz` files for Working Memory, Fear, Neutral, and Language tasks are stored in a folder with this path: `./ref_activation_ns/data/NS_ref_activation_data`

In [1]:
!pip install nilearn --quiet
!pip install nibabel --quiet
!pip install seaborn --quiet
!pip install brainspace --quiet

import os
import tempfile
import requests
import numpy as np
import nibabel as nib
import seaborn as sns
import antspyx as ants
import matplotlib.pyplot as plt

from nilearn import plotting, datasets
from brainspace.utils.parcellation import map_to_labels

ModuleNotFoundError: No module named 'antspyx'

Next, navigate to the main directory and run this code in your terminal:
```bash
fslmaths ./ref_activation_ns/data/mni_icbm152_nlin_asym_09a_nifti/mni_icbm152_nlin_asym_09a/mni_icbm152_wm_tal_nlin_asym_09a.nii -mul 20 -add ./ref_activation_ns/data/mni_icbm152_nlin_asym_09a_nifti/mni_icbm152_nlin_asym_09a/mni_icbm152_pd_tal_nlin_asym_09a.nii ./ref_activation_ns/data/mni_icbm152_asym_A.nii.gz

```

Next, run these commands on the atlas if you want to use `fastsurfer`:
```bash
# Number of threads
threads=15
# Image path
fastsurfer_img=fastsurfer-cpu-v2.2.0.sif
# Temporary directory path
export TMPDIR=/tmp/tmpfiles
# Freesurfer license
fs_license=/freesurfer-7.3.2/license.txt
# Path to outputs
SUBJECTS_DIR=/out/fastsurfer
# Subject ID
idBIDS=mni_icbm152_asym_C
# MRI|IMG to process
t1=mni_icbm152_asym_C.nii.gz

# Run the singularity container
singularity exec --writable-tmpfs --containall \
             -B "${SUBJECTS_DIR}":/output \
             -B "${TMPDIR}":/tmpdir \
             -B "${t1}":/tmpdir/${idBIDS}_T1w.nii.gz \
             -B "${fs_license}":/output/license.txt \
             "${fastsurfer_img}" \
                 /fastsurfer/run_fastsurfer.sh \
                 --fs_license /output/license.txt \
                 --t1 /tmpdir/${idBIDS}_T1w.nii.gz \
                 --sid "${idBIDS}" --sd /output --no_fs_T1 \
                 --parallel --threads "${threads}"
```

Alternatively, run these codes to use `freesurfer` or `fsl`'s `FIRST`:
- For `freesurfer`:
```bash
# Install FreeSurfer (if not already installed)
# Set FreeSurfer environment variables
export FREESURFER_HOME=/usr/local/freesurfer # or ~/freesurfer/ for Linux
source $FREESURFER_HOME/SetUpFreeSurfer.sh

# Path to the input T1-weighted image (from main project directory)
t1=./ref_activation_ns/data/mni_icbm152_asym_A.nii.gz

# Path to the output directory (from main project directory)
SUBJECTS_DIR=./ref_activation_ns/data/out/freesurfer

# Subject ID
idBIDS=mni_icbm152_asym_C

# Run FreeSurfer's recon-all
recon-all -i "$t1" -s "$idBIDS" -sd "$SUBJECTS_DIR" -all
```

- For `fsl`'s `FIRST`:
```bash
# Path to the input T1-weighted image
t1=./ref_activation_ns/data/mni_icbm152_asym_A.nii.gz

# Path to the output directory
OUTDIR=./ref_activation_ns/data/out/first

# Subject ID
idBIDS=mni_icbm152_asym_C

# Run FSL's run_first_all
run_first_all -i "$t1" -o "$OUTDIR/$idBIDS" -s "$idBIDS"
```

### Download the `anatomical.nii.gz` files from Neurosynth
We downloaded `association-test_z_FDR_0.01.nii.gz` files corresponding to our target tasks from the following links:
* [Working Memory](https://neurosynth.org/analyses/terms/working%20memory/)
* Emotion subtasks:
  * [Neutral](https://neurosynth.org/analyses/terms/neutral/)
  * [Fear](https://neurosynth.org/analyses/terms/fear/)
* [Language](https://neurosynth.org/analyses/terms/language/)

### Registraction with `fsl`'s `flirt` instead of `mri_easyreg` from `fastsurfer`

```bash
# Path to the MNI template
atlas_mni152="./ref_activation_ns/data/mni_icbm152_asym_A.nii.gz"

# Path to the Neurosynth anatomical images
neurosynth_anat="./ref_activation_ns/data/NS_ref_activation_data/working memory_association-test_z_FDR_0.01.nii.gz" #For working memory

# Create and give permissions to directories
mkdir -p ./ref_activation_ns/data/out/neurosynth/
mkdir -p neurosynth/
chmod u+w ./ref_activation_ns/data/out/neurosynth/
chmod u+w neurosynth/

#Path to the output tranformation matrix
xfm_mat="./ref_activation_ns/data/out/neurosynth/from-neurosynth_to-mniICBM152_asym_C_desc-flirt.mat"

# Perform linear registration using flirt
flirt -in "$neurosynth_anat" -ref "$atlas_mni152" -out "neurosynth/anatomical_space-mni_icbm152_asym_C" -omat "$xfm_mat"
```

Run the same command for other subtasks, modifying the `neurosynth_anat` and `xfm_mat` each time.

### Registration via ANTs anatomical.nii.gz to mni_icbm152_asym_C.nii.gz (Image-based and Label-based)

```bash
# Path to the MNI template
atlas_mni152="./ref_activation_ns/data/mni_icbm152_asym_A.nii.gz"

# Path to the Neurosynth anatomical image
neurosynth_anat="./ref_activation_ns/data/NS_ref_activation_data/working memory_association-test_z_FDR_0.01.nii.gz" #For working memory

# Path to the output transformation files
xfm_prefix=./ref_activation_ns/data/out/neurosynth/from-neurosynth_to-mniICBM152_asym_C_desc-SyN_

# Perform SyN registration using antsRegistrationSyN.sh
antsRegistrationSyN.sh -d 3 -f "$atlas_mni152" -m "$neurosynth_anat" -o "$xfm_prefix" -t s -n 30 -p d
```

#### Label-based registration using `antspyx`

In [ ]:
# The path to the MNI template
atlas_mni152_path = "./ref_activation_ns/data/mni_icbm152_asym_A.nii.gz"

# The path to the Neurosynth anatomical image (for working memory)
neurosynth_anat_path = "./ref_activation_ns/data/NS_ref_activation_data/working memory_association-test_z_FDR_0.01.nii.gz"

# The prefix for the output transformation files (used for saving)
output_prefix = "./ref_activation_ns/data/out/neurosynth/from-neurosynth_to-mniICBM152_asym_C_desc-SyN_"

# Load the images using ants.image_read
fixed_image = ants.image_read(atlas_mni152_path)
moving_image = ants.image_read(neurosynth_anat_path)

# Perform SyN registration using ants.registration
registration = ants.registration(
    fixed = fixed_image,
    moving = moving_image,
    type_of_transform = "SyN",
    iterations = (30, 30, 20),
    convergence_threshold = [1e-6, 1e-6, 1e-6],
    sigma_gradient = [1.0, 0.5, 0.0],
    aff_metric = "mattes",
    aff_sampling = 32,
    aff_random_sampling_rate = 0.5,
    transforms = ("Rigid", "Affine", "SyN"),
    transform_parameters = ((0.1,), (0.1,), (0.1, 3.0, 0.0)),
    metric = ("MI", "MI", "CC"),
    metric_weight = (1, 1, 1),
    sampling_strategy = ("Regular", "Regular", None),
    sampling_percentage = (0.25, 0.25, None),
    radius = (None, None, 4),
    number_of_affine_iterations = (10000, 10000, 10000),
    initial_moving_transform = None,
    interpolation = "linear",
    verbose = True
)

# Extract the transformation matrices and warped moving image
warp_transform = registration["warp"]
affine_transform = registration["affine"]
warped_moving_image = registration["warpedmovout"]

# Save the transformation matrices and the warped image
ants.image_write(warp_transform, output_prefix + "Warp.nii.gz")
ants.image_write(affine_transform, output_prefix + "Affine.mat")
ants.image_write(warped_moving_image, output_prefix + "Warped.nii.gz")

print(f"Warp field saved to: {output_prefix}Warp.nii.gz")
print(f"Affine transform saved to: {output_prefix}Affine.mat")
print(f"Warped moving image saved to: {output_prefix}Warped.nii.gz")